# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a template for loading and exploring a dataset using the [`mlcroissant`](https://pypi.org/project/mlcroissant/) library.

### Dataset Source
The dataset source is provided via a [Croissant schema URL](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json).

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Instantiate the dataset
dataset = mlc.Dataset(croissant_url)

# Access and display metadata
meta = dataset.metadata
print(f"Name: {meta.name}\nDescription: {meta.description}\nVersion: {getattr(meta, 'version', 'unknown')}")

## 2. Data Overview
Review available record sets, their fields, and collect `@id`s for further processing.

A record set represents a table-like collection of records described in the Croissant schema. We'll enumerate them and explore their fields.

In [ ]:
record_sets = list(dataset.record_sets)
print(f"Found {len(record_sets)} record set(s):\n")
record_sets_ids = []
for rec in record_sets:
    print(f"- RecordSet Name: {rec.name}")
    print(f"  @id: {rec.id}")
    print(f"  Description: {getattr(rec, 'description', 'No description available.')}")
    print(f"  Fields (@id and name):")
    for field in rec.fields:
        print(f"    - {field.id} : {field.name} (dataType: {getattr(field, 'data_type', 'unknown')})")
    print()
    record_sets_ids.append(rec.id)
# Save for later usage
if record_sets_ids:
    primary_record_set_id = record_sets_ids[0]

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Here, we demonstrate extracting the records from the primary record set identified above.

We'll use the record set `@id` and the field `@id`s for referencing, as required for reproducibility and schema clarity.

In [ ]:
# Prepare a dictionary to hold DataFrames for each record set
dataframes = dict()
for record_set_id in record_sets_ids:
    print(f"Loading records from record set @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"  -> Loaded {len(df)} records. Columns: {df.columns.tolist()[:5]} ...\n")

# Preview primary record set (first 5 rows)
main_df = dataframes[primary_record_set_id]
print(f"Columns in primary record set ({primary_record_set_id}):\n{main_df.columns.tolist()}")
main_df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Below, we demonstrate how to:
- Select a numeric field (by its `@id`),
- Filter for values above a chosen threshold,
- Normalize that field,
- Group by a categorical field if available.

In [ ]:
# Identify numeric and categorical fields for demonstration
fields = {field.id: field for field in next(iter(dataset.record_sets)).fields}

# Try to select a numeric field
numeric_field_id = None
for fid, field in fields.items():
    if getattr(field, "data_type", None) in ["Float", "Number", "Integer"]:
        numeric_field_id = fid
        break
if numeric_field_id is None:
    raise ValueError("No numeric field found in this record set!")
print(f"Selected numeric field: {numeric_field_id}")

# Convert the numeric field to float and handle possible conversion errors
main_df[numeric_field_id] = pd.to_numeric(main_df[numeric_field_id], errors='coerce')

threshold = main_df[numeric_field_id].mean() if main_df[numeric_field_id].notnull().any() else 10
filtered_df = main_df[main_df[numeric_field_id] > threshold].copy()
print(f"Filtered records where {numeric_field_id} > {threshold:.2f}: {len(filtered_df)} records\n")

# Normalizing the numeric field
if filtered_df[numeric_field_id].notnull().sum() > 1:
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
        filtered_df[numeric_field_id].std()
    )
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
else:
    print("Not enough data to normalize.")

# Optionally group by a categorical field for summary stats
categorical_field_id = None
for fid, field in fields.items():
    if getattr(field, "data_type", None) == "Text" and fid != numeric_field_id:
        categorical_field_id = fid
        break
if categorical_field_id is not None and categorical_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(categorical_field_id)[numeric_field_id].mean().reset_index()
    print(f"Mean {numeric_field_id} grouped by {categorical_field_id} (first 5 groups):\n")
    print(grouped_df.head())
else:
    print("No suitable categorical field for grouping found.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Let's create a histogram of the selected numeric field and, if available, a boxplot grouped by the selected categorical field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram for the numeric field
plt.figure(figsize=(8, 4))
main_df[numeric_field_id].dropna().hist(bins=30)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.show()

# Boxplot if a categorical field is available
if categorical_field_id and categorical_field_id in main_df.columns:
    plt.figure(figsize=(10, 5))
    sns.boxplot(x=categorical_field_id, y=numeric_field_id, data=main_df)
    plt.title(f"{numeric_field_id} distribution grouped by {categorical_field_id}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion

In this notebook, we demonstrated how to load, inspect, and preprocess a scientific mass spectrometry dataset using the `mlcroissant` library and referenced all entities by their Croissant `@id` fields. Key findings:
- Explored the dataset structure, fields, and metadata using Croissant schema.
- Filtered and normalized a numeric field, illustrating typical FAIR data workflows.
- Visualized core data distributions with matplotlib and seaborn.

For more advanced analyses or machine learning, consider deeper feature engineering using field metadata.
